<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/model-train-comparison/notebooks/06_best_model_selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Best Model Selection

#Setup, Drive Mount, Paths, Output Folders

In [ ]:
# Setup with fresh Drive mount

from google.colab import drive
from pathlib import Path
from datetime import datetime
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt

drive.mount("/content/gdrive")

base_dir = Path("/content/gdrive/MyDrive/USW_Dissertation/Prompt-Injection-Detection-System-SHAP")

model_dir = base_dir / "models"
result_dir = base_dir / "results"
notebook_dir = base_dir / "notebooks"

compare_dir = result_dir / "model_comparison"
table_dir = compare_dir / "tables"
fig_dir = compare_dir / "figures"
log_dir = compare_dir / "logs"
config_dir = compare_dir / "configs"

models = ["distilbert", "roberta", "secbert"]

for folder in [base_dir, model_dir, result_dir, notebook_dir]:
    if not folder.exists():
        raise FileNotFoundError(f"Missing folder: {folder}")

for folder in [compare_dir, table_dir, fig_dir, log_dir, config_dir]:
    folder.mkdir(parents=True, exist_ok=True)

setup_check = pd.DataFrame([
    {"item": "base_dir", "path": str(base_dir), "found": base_dir.exists()},
    {"item": "model_dir", "path": str(model_dir), "found": model_dir.exists()},
    {"item": "result_dir", "path": str(result_dir), "found": result_dir.exists()},
    {"item": "notebook_dir", "path": str(notebook_dir), "found": notebook_dir.exists()},
    {"item": "compare_dir", "path": str(compare_dir), "found": compare_dir.exists()},
    {"item": "table_dir", "path": str(table_dir), "found": table_dir.exists()},
    {"item": "fig_dir", "path": str(fig_dir), "found": fig_dir.exists()},
    {"item": "log_dir", "path": str(log_dir), "found": log_dir.exists()},
    {"item": "config_dir", "path": str(config_dir), "found": config_dir.exists()}
])

setup_file = table_dir / "setup_check.csv"
setup_check.to_csv(setup_file, index=False)

display(setup_check)
print("Saved:", setup_file)

#Part 2 — Check Model and Result Files

In [ ]:
# Part 2 - check model and result files

def find_file(words, exts=(".csv", ".json")):
    for file in result_dir.rglob("*"):
        if not file.is_file():
            continue

        text = str(file).lower()

        if file.suffix.lower() not in exts:
            continue

        if all(word in text for word in words):
            return file

    return None


rows = []
file_rows = []

for model in models:
    model_path = model_dir / model
    best_path = model_path / "best_model"
    tok_path = model_path / "tokenizer"

    tok_found = tok_path.exists()

    if not tok_found and best_path.exists():
        tok_files = [
            best_path / "tokenizer.json",
            best_path / "tokenizer_config.json",
            best_path / "vocab.txt",
            best_path / "merges.txt"
        ]
        tok_found = any(file.exists() for file in tok_files)

    val_metrics = find_file([model, "validation", "metric"])
    if val_metrics is None:
        val_metrics = find_file([model, "val", "metric"])

    test_metrics = find_file([model, "test", "metric"])
    test_pred = find_file([model, "test", "prediction"])
    conf_mat = find_file([model, "confusion"])
    false_pos = find_file([model, "false", "positive"])
    false_neg = find_file([model, "false", "negative"])
    train_log = find_file([model, "trainer", "log", "history"])

    if train_log is None:
        train_log = find_file([model, "training", "metric"])

    found_files = {
        "validation_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_predictions": test_pred,
        "confusion_matrix": conf_mat,
        "false_positives": false_pos,
        "false_negatives": false_neg,
        "training_log": train_log
    }

    rows.append({
        "model": model,
        "model_folder": model_path.exists(),
        "best_model": best_path.exists(),
        "tokenizer": tok_found,
        "validation_metrics": val_metrics is not None,
        "test_metrics": test_metrics is not None,
        "test_predictions": test_pred is not None,
        "confusion_matrix": conf_mat is not None,
        "false_positives": false_pos is not None,
        "false_negatives": false_neg is not None,
        "training_log": train_log is not None
    })

    file_rows.append({
        "model": model,
        "file_type": "model_folder",
        "path": str(model_path),
        "found": model_path.exists()
    })

    file_rows.append({
        "model": model,
        "file_type": "best_model",
        "path": str(best_path),
        "found": best_path.exists()
    })

    file_rows.append({
        "model": model,
        "file_type": "tokenizer",
        "path": str(tok_path),
        "found": tok_found
    })

    for file_type, file_path in found_files.items():
        file_rows.append({
            "model": model,
            "file_type": file_type,
            "path": "" if file_path is None else str(file_path),
            "found": file_path is not None
        })


file_check = pd.DataFrame(rows)
file_manifest = pd.DataFrame(file_rows)

file_check_path = table_dir / "file_availability_check.csv"
manifest_path = table_dir / "located_files_manifest.csv"

file_check.to_csv(file_check_path, index=False)
file_manifest.to_csv(manifest_path, index=False)

display(file_check)

print("Saved:", file_check_path)
print("Saved:", manifest_path)

#Part 3 — Load Validation Metrics

In [ ]:
# load validation metrics

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

def get_metric(df, names):
    cols = {col.lower().strip(): col for col in df.columns}

    for name in names:
        if name in cols:
            return df[cols[name]].iloc[0]

    if "metric" in cols and "value" in cols:
        metric_col = cols["metric"]
        value_col = cols["value"]

        for name in names:
            match = df[df[metric_col].astype(str).str.lower().str.strip() == name]
            if len(match) > 0:
                return match[value_col].iloc[0]

    return np.nan


rows = []

for model in models:
    file_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "validation_metrics")
    ]["path"].iloc[0]

    metric_df = pd.read_csv(file_path)

    rows.append({
        "model": model,
        "validation_accuracy": get_metric(metric_df, ["accuracy", "eval_accuracy", "val_accuracy"]),
        "validation_precision": get_metric(metric_df, ["precision", "eval_precision", "val_precision"]),
        "validation_recall": get_metric(metric_df, ["recall", "eval_recall", "val_recall"]),
        "validation_f1": get_metric(metric_df, ["f1", "eval_f1", "val_f1"]),
        "validation_auc_roc": get_metric(metric_df, ["auc_roc", "roc_auc", "eval_auc_roc", "eval_auc"])
    })

val_df = pd.DataFrame(rows)

for col in val_df.columns:
    if col != "model":
        val_df[col] = pd.to_numeric(val_df[col], errors="coerce")

val_file = table_dir / "validation_metrics_comparison.csv"
val_df.to_csv(val_file, index=False)

display(val_df)
print("Saved:", val_file)

#Part 4 — Load Test Metrics

In [ ]:
# load test metrics

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

rows = []

for model in models:
    file_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "test_metrics")
    ]["path"].iloc[0]

    metric_df = pd.read_csv(file_path)

    rows.append({
        "model": model,
        "test_accuracy": get_metric(metric_df, ["accuracy", "eval_accuracy", "test_accuracy"]),
        "test_precision": get_metric(metric_df, ["precision", "eval_precision", "test_precision"]),
        "test_recall": get_metric(metric_df, ["recall", "eval_recall", "test_recall"]),
        "test_f1": get_metric(metric_df, ["f1", "eval_f1", "test_f1"]),
        "test_auc_roc": get_metric(metric_df, ["auc_roc", "roc_auc", "eval_auc_roc", "eval_auc"])
    })

test_df = pd.DataFrame(rows)

for col in test_df.columns:
    if col != "model":
        test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

test_file = table_dir / "test_metrics_comparison.csv"
test_df.to_csv(test_file, index=False)

display(test_df)
print("Saved:", test_file)

#Part 5 — Load Training Logs

In [ ]:
# load training logs

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

rows = []
log_list = []

for model in models:
    log_file = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "training_log")
    ]["path"].iloc[0]

    if log_file.endswith(".json"):
        with open(log_file, "r") as f:
            data = json.load(f)
        log_df = pd.DataFrame(data["log_history"])
    else:
        log_df = pd.read_csv(log_file)

    log_df["model"] = model
    log_list.append(log_df)

    f1_col = None
    for col in ["eval_f1", "validation_f1", "val_f1", "f1"]:
        if col in log_df.columns:
            f1_col = col
            break

    if f1_col is not None:
        valid_f1 = log_df[log_df[f1_col].notna()]
        best_row = valid_f1.loc[valid_f1[f1_col].idxmax()]
        best_f1 = best_row[f1_col]
        best_epoch = best_row["epoch"] if "epoch" in log_df.columns else np.nan
    else:
        best_f1 = np.nan
        best_epoch = np.nan

    if "loss" in log_df.columns and log_df["loss"].notna().sum() > 0:
        train_loss = log_df["loss"].dropna().iloc[-1]
    elif "train_loss" in log_df.columns and log_df["train_loss"].notna().sum() > 0:
        train_loss = log_df["train_loss"].dropna().iloc[-1]
    else:
        train_loss = np.nan

    if "eval_loss" in log_df.columns and log_df["eval_loss"].notna().sum() > 0:
        eval_loss = log_df["eval_loss"].dropna().iloc[-1]
    else:
        eval_loss = np.nan

    if "train_runtime" in log_df.columns and log_df["train_runtime"].notna().sum() > 0:
        runtime = log_df["train_runtime"].dropna().iloc[-1]
    else:
        runtime = np.nan

    rows.append({
        "model": model,
        "log_rows": len(log_df),
        "best_epoch": best_epoch,
        "best_validation_f1": best_f1,
        "final_train_loss": train_loss,
        "final_eval_loss": eval_loss,
        "train_runtime": runtime
    })

train_df = pd.DataFrame(rows)
all_logs = pd.concat(log_list, ignore_index=True)

train_file = table_dir / "training_stability_summary.csv"
all_log_file = table_dir / "all_training_logs.csv"

train_df.to_csv(train_file, index=False)
all_logs.to_csv(all_log_file, index=False)

display(train_df)
print("Saved:", train_file)

#Part 6 — Load Test Predictions

In [ ]:

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

def pick_col(df, names):
    cols = {col.lower(): col for col in df.columns}

    for name in names:
        if name.lower() in cols:
            return cols[name.lower()]

    return None


def find_prob_cols(df, used_cols):
    safe_col = pick_col(df, [
        "probability_SAFE", "prob_safe", "safe_prob", "safe_probability",
        "probability_0", "prob_0", "class_0_prob", "score_0"
    ])

    inject_col = pick_col(df, [
        "probability_INJECTION", "prob_injection", "injection_prob", "injection_probability",
        "probability_1", "prob_1", "class_1_prob", "score_1"
    ])

    if safe_col is not None and inject_col is not None:
        return safe_col, inject_col

    prob_cols = []

    for col in df.columns:
        if col in used_cols:
            continue

        col_name = col.lower()
        values = pd.to_numeric(df[col], errors="coerce")

        if values.notna().sum() == len(df):
            if values.between(0, 1).all():
                if "prob" in col_name or "score" in col_name or "confidence" in col_name:
                    prob_cols.append(col)

    if len(prob_cols) >= 2:
        return prob_cols[0], prob_cols[1]

    return safe_col, inject_col


pred_data = {}
rows = []

base_ids = None
base_labels = None

for model in models:
    file_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "test_predictions")
    ]["path"].iloc[0]

    df = pd.read_csv(file_path)

    text_col = pick_col(df, ["text_for_model", "text", "prompt"])
    id_col = pick_col(df, ["id", "index", "sample_id"])
    true_col = pick_col(df, ["true_label", "label", "y_true"])
    pred_col = pick_col(df, ["predicted_label", "prediction", "y_pred"])

    if id_col is None:
        df["id"] = range(len(df))
        id_col = "id"

    used_cols = [text_col, id_col, true_col, pred_col]
    used_cols = [col for col in used_cols if col is not None]

    safe_col, inject_col = find_prob_cols(df, used_cols)

    ids = list(df[id_col])
    labels = list(df[true_col])

    if base_ids is None:
        base_ids = ids
        base_labels = labels
        same_rows = True
        same_labels = True
    else:
        same_rows = ids == base_ids
        same_labels = labels == base_labels

    note = ""
    if safe_col is None or inject_col is None:
        note = "probability columns not found"

    pred_data[model] = df

    rows.append({
        "model": model,
        "rows": len(df),
        "text_col": text_col,
        "id_col": id_col,
        "true_col": true_col,
        "pred_col": pred_col,
        "safe_prob_col": safe_col,
        "inject_prob_col": inject_col,
        "same_rows": same_rows,
        "same_labels": same_labels,
        "note": note,
        "file_path": file_path
    })

pred_check = pd.DataFrame(rows)

pred_file = table_dir / "prediction_file_check_summary.csv"
pred_check.to_csv(pred_file, index=False)

display(pred_check.drop(columns=["file_path"]))

print("Saved:", pred_file)

## Part 7 - confusion matrix comparison

In [ ]:

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")
pred_check = pd.read_csv(table_dir / "prediction_file_check_summary.csv")

def to_label(value):
    text = str(value).strip().lower()

    if text in ["0", "0.0", "safe", "benign"]:
        return 0

    if text in ["1", "1.0", "injection", "attack", "malicious"]:
        return 1

    return np.nan


rows = []

for _, item in pred_check.iterrows():
    model = item["model"]

    file_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "test_predictions")
    ]["path"].iloc[0]

    df = pd.read_csv(file_path)

    true_col = item["true_col"]
    pred_col = item["pred_col"]

    y_true = df[true_col].apply(to_label)
    y_pred = df[pred_col].apply(to_label)

    if y_true.isna().sum() > 0 or y_pred.isna().sum() > 0:
        raise ValueError(f"Label conversion problem in {model}")

    tn = ((y_true == 0) & (y_pred == 0)).sum()
    fp = ((y_true == 0) & (y_pred == 1)).sum()
    fn = ((y_true == 1) & (y_pred == 0)).sum()
    tp = ((y_true == 1) & (y_pred == 1)).sum()

    safe_count = (y_true == 0).sum()
    inject_count = (y_true == 1).sum()

    rows.append({
        "model": model,
        "total": len(df),
        "safe_count": safe_count,
        "injection_count": inject_count,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
        "false_positives": fp,
        "false_negatives": fn,
        "fp_rate": round(fp / safe_count, 4),
        "fn_rate": round(fn / inject_count, 4)
    })

conf_df = pd.DataFrame(rows)

conf_file = table_dir / "confusion_matrix_comparison.csv"
conf_df.to_csv(conf_file, index=False)

display(conf_df)
print("Saved:", conf_file)

# Part 8 - Error and Confidence analysis

In [ ]:

file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")
pred_check = pd.read_csv(table_dir / "prediction_file_check_summary.csv")

def clean_label(value):
    text = str(value).strip().lower()

    if text in ["0", "0.0", "safe", "benign"]:
        return 0

    if text in ["1", "1.0", "injection", "attack", "malicious"]:
        return 1

    return np.nan


pred_rows = []

for _, item in pred_check.iterrows():
    model = item["model"]

    file_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "test_predictions")
    ]["path"].iloc[0]

    df = pd.read_csv(file_path)

    id_col = item["id_col"]
    text_col = item["text_col"]
    true_col = item["true_col"]
    pred_col = item["pred_col"]
    safe_col = item["safe_prob_col"]
    inject_col = item["inject_prob_col"]

    for _, row in df.iterrows():
        true_label = clean_label(row[true_col])
        pred_label = clean_label(row[pred_col])

        safe_prob = float(row[safe_col])
        inject_prob = float(row[inject_col])
        confidence = max(safe_prob, inject_prob)

        if true_label == pred_label:
            error_type = "correct"
        elif true_label == 0 and pred_label == 1:
            error_type = "false_positive"
        else:
            error_type = "false_negative"

        pred_rows.append({
            "model": model,
            "id": row[id_col],
            "text": row[text_col],
            "true_label": int(true_label),
            "predicted_label": int(pred_label),
            "safe_probability": safe_prob,
            "injection_probability": inject_prob,
            "confidence": confidence,
            "is_correct": true_label == pred_label,
            "error_type": error_type
        })

all_pred = pd.DataFrame(pred_rows)
errors = all_pred[all_pred["is_correct"] == False].copy()

rows = []

for model in models:
    model_df = all_pred[all_pred["model"] == model]
    error_df = errors[errors["model"] == model]

    rows.append({
        "model": model,
        "total_predictions": len(model_df),
        "correct_predictions": model_df["is_correct"].sum(),
        "total_errors": len(error_df),
        "false_positives": (error_df["error_type"] == "false_positive").sum(),
        "false_negatives": (error_df["error_type"] == "false_negative").sum(),
        "mean_correct_confidence": model_df[model_df["is_correct"]]["confidence"].mean(),
        "mean_wrong_confidence": error_df["confidence"].mean(),
        "high_confidence_wrong": (error_df["confidence"] >= 0.90).sum()
    })

error_summary = pd.DataFrame(rows)

common_errors = (
    errors
    .groupby(["id", "text", "true_label"])
    .agg(
        models_wrong=("model", lambda x: ", ".join(x)),
        wrong_model_count=("model", "count")
    )
    .reset_index()
    .sort_values(["wrong_model_count", "id"], ascending=[False, True])
)

shared_fn = errors[errors["error_type"] == "false_negative"].copy()

shared_fn = (
    shared_fn
    .groupby(["id", "text", "true_label"])
    .agg(
        models_missed=("model", lambda x: ", ".join(x)),
        missed_model_count=("model", "count")
    )
    .reset_index()
    .sort_values(["missed_model_count", "id"], ascending=[False, True])
)

high_wrong = errors[errors["confidence"] >= 0.90].copy()

all_pred_file = table_dir / "standardized_test_predictions.csv"
error_file = table_dir / "all_model_errors.csv"
summary_file = table_dir / "error_comparison_summary.csv"
common_file = table_dir / "common_error_analysis.csv"
shared_fn_file = table_dir / "shared_false_negatives.csv"
high_wrong_file = table_dir / "high_confidence_wrong_predictions.csv"

all_pred.to_csv(all_pred_file, index=False)
errors.to_csv(error_file, index=False)
error_summary.to_csv(summary_file, index=False)
common_errors.to_csv(common_file, index=False)
shared_fn.to_csv(shared_fn_file, index=False)
high_wrong.to_csv(high_wrong_file, index=False)

display(error_summary)

print("Saved:", summary_file)

# Part 9 - Model Comparison

In [ ]:

val_df = pd.read_csv(table_dir / "validation_metrics_comparison.csv")
test_df = pd.read_csv(table_dir / "test_metrics_comparison.csv")
train_df = pd.read_csv(table_dir / "training_stability_summary.csv")
conf_df = pd.read_csv(table_dir / "confusion_matrix_comparison.csv")
error_df = pd.read_csv(table_dir / "error_comparison_summary.csv")
file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

summary = val_df.merge(test_df, on="model", how="left")
summary = summary.merge(train_df, on="model", how="left")

conf_cols = [
    "model", "total", "safe_count", "injection_count",
    "tn", "fp", "fn", "tp", "fp_rate", "fn_rate"
]

summary = summary.merge(conf_df[conf_cols], on="model", how="left")

error_cols = [
    "model", "correct_predictions", "total_errors",
    "mean_correct_confidence", "mean_wrong_confidence",
    "high_confidence_wrong"
]

summary = summary.merge(error_df[error_cols], on="model", how="left")

path_rows = []

for model in models:
    model_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "model_folder")
    ]["path"].iloc[0]

    best_model_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "best_model")
    ]["path"].iloc[0]

    tokenizer_path = file_manifest[
        (file_manifest["model"] == model) &
        (file_manifest["file_type"] == "tokenizer")
    ]["path"].iloc[0]

    path_rows.append({
        "model": model,
        "model_path": model_path,
        "best_model_path": best_model_path,
        "tokenizer_path": tokenizer_path
    })

path_df = pd.DataFrame(path_rows)
summary = summary.merge(path_df, on="model", how="left")

summary = summary[[
    "model",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_auc_roc",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auc_roc",
    "tn",
    "fp",
    "fn",
    "tp",
    "fp_rate",
    "fn_rate",
    "correct_predictions",
    "total_errors",
    "mean_correct_confidence",
    "mean_wrong_confidence",
    "high_confidence_wrong",
    "best_epoch",
    "best_validation_f1",
    "final_train_loss",
    "final_eval_loss",
    "train_runtime",
    "model_path",
    "best_model_path",
    "tokenizer_path"
]]

summary_file = table_dir / "model_comparison_summary.csv"
summary.to_csv(summary_file, index=False)

display(summary)

print("Saved:", summary_file)

# Part 10 - Model Ranking

In [ ]:
# Part 10 - transparent model ranking and recommendation

summary = pd.read_csv(table_dir / "model_comparison_summary.csv")

decision_cols = [
    "model",
    "validation_accuracy",
    "validation_precision",
    "validation_recall",
    "validation_f1",
    "validation_auc_roc",
    "test_accuracy",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auc_roc",
    "fp",
    "fn",
    "total_errors",
    "high_confidence_wrong",
    "final_eval_loss"
]

decision_df = summary[decision_cols].copy()

decision_df["fn_rank"] = decision_df["fn"].rank(method="min", ascending=True)
decision_df["fp_rank"] = decision_df["fp"].rank(method="min", ascending=True)
decision_df["test_recall_rank"] = decision_df["test_recall"].rank(method="min", ascending=False)
decision_df["test_precision_rank"] = decision_df["test_precision"].rank(method="min", ascending=False)
decision_df["test_f1_rank"] = decision_df["test_f1"].rank(method="min", ascending=False)
decision_df["test_auc_rank"] = decision_df["test_auc_roc"].rank(method="min", ascending=False)
decision_df["validation_accuracy_rank"] = decision_df["validation_accuracy"].rank(method="min", ascending=False)
decision_df["validation_f1_rank"] = decision_df["validation_f1"].rank(method="min", ascending=False)
decision_df["error_rank"] = decision_df["total_errors"].rank(method="min", ascending=True)

decision_df["security_score"] = (
    decision_df["fn_rank"] * 3 +
    decision_df["test_recall_rank"] * 2 +
    decision_df["test_f1_rank"] * 2 +
    decision_df["test_auc_rank"] +
    decision_df["fp_rank"] +
    decision_df["error_rank"]
)

decision_df["overall_score"] = (
    decision_df["security_score"] +
    decision_df["validation_f1_rank"] +
    decision_df["validation_accuracy_rank"]
)

decision_df = decision_df.sort_values(
    ["overall_score", "security_score", "fn", "test_f1"],
    ascending=[True, True, True, False]
).reset_index(drop=True)

decision_df["final_rank"] = decision_df.index + 1
selected_model = decision_df.loc[0, "model"]

decision_df["recommended_for_shap_webapp"] = decision_df["model"] == selected_model

decision_df["decision_reason"] = ""

for idx, row in decision_df.iterrows():
    if row["model"] == selected_model:
        decision_df.loc[idx, "decision_reason"] = (
            "Selected because it has the strongest deployment/security profile: "
            "lowest false negatives, strongest test F1, test recall, AUC-ROC, "
            "and lowest total errors."
        )
    elif row["model"] == decision_df.sort_values("validation_f1", ascending=False).iloc[0]["model"]:
        decision_df.loc[idx, "decision_reason"] = (
            "Best validation model, but weaker on held-out test/security errors."
        )
    else:
        decision_df.loc[idx, "decision_reason"] = (
            "Not selected because test/security performance is weaker than the recommended model."
        )

rule_df = pd.DataFrame([
    {
        "priority": 1,
        "metric": "False negatives",
        "column": "fn",
        "direction": "lower is better",
        "reason": "A missed INJECTION prompt is the highest security risk."
    },
    {
        "priority": 2,
        "metric": "Test recall",
        "column": "test_recall",
        "direction": "higher is better",
        "reason": "Shows how well the model catches actual INJECTION prompts."
    },
    {
        "priority": 3,
        "metric": "Test F1-score",
        "column": "test_f1",
        "direction": "higher is better",
        "reason": "Balances precision and recall on the held-out test set."
    },
    {
        "priority": 4,
        "metric": "Test AUC-ROC",
        "column": "test_auc_roc",
        "direction": "higher is better",
        "reason": "Shows class separation quality across thresholds."
    },
    {
        "priority": 5,
        "metric": "False positives",
        "column": "fp",
        "direction": "lower is better",
        "reason": "Reduces wrongly blocking SAFE prompts."
    },
    {
        "priority": 6,
        "metric": "Test precision",
        "column": "test_precision",
        "direction": "higher is better",
        "reason": "Shows how reliable INJECTION predictions are."
    },
    {
        "priority": 7,
        "metric": "Validation F1-score",
        "column": "validation_f1",
        "direction": "higher is better",
        "reason": "Shows validation-stage model quality."
    },
    {
        "priority": 8,
        "metric": "Validation accuracy",
        "column": "validation_accuracy",
        "direction": "higher is better",
        "reason": "Used as supporting validation evidence, not the main security criterion."
    }
])

recommendation = pd.DataFrame([
    {
        "selected_model": selected_model,
        "validation_best_model": decision_df.sort_values("validation_f1", ascending=False).iloc[0]["model"],
        "test_f1_best_model": decision_df.sort_values("test_f1", ascending=False).iloc[0]["model"],
        "test_auc_best_model": decision_df.sort_values("test_auc_roc", ascending=False).iloc[0]["model"],
        "lowest_fn_model": decision_df.sort_values("fn", ascending=True).iloc[0]["model"],
        "lowest_fp_model": decision_df.sort_values("fp", ascending=True).iloc[0]["model"],
        "final_reason": "Recommended model is selected using FN, FP, recall, precision, F1, AUC-ROC, validation F1, and validation accuracy."
    }
])

decision_file = table_dir / "transparent_model_selection_decision.csv"
rule_file = table_dir / "model_selection_rule.csv"
recommend_file = table_dir / "recommended_model_for_shap.csv"

decision_df.to_csv(decision_file, index=False)
rule_df.to_csv(rule_file, index=False)
recommendation.to_csv(recommend_file, index=False)

display(decision_df[[
    "model",
    "final_rank",
    "validation_accuracy",
    "validation_f1",
    "test_precision",
    "test_recall",
    "test_f1",
    "test_auc_roc",
    "fp",
    "fn",
    "total_errors",
    "security_score",
    "overall_score",
    "recommended_for_shap_webapp",
    "decision_reason"
]])

display(rule_df)
display(recommendation)

print("Saved:", decision_file)
print("Saved:", rule_file)
print("Saved:", recommend_file)

# Part 11 - Model configs

In [ ]:

recommendation = pd.read_csv(table_dir / "recommended_model_for_shap.csv")
summary = pd.read_csv(table_dir / "model_comparison_summary.csv")
pred_check = pd.read_csv(table_dir / "prediction_file_check_summary.csv")
file_manifest = pd.read_csv(table_dir / "located_files_manifest.csv")

selected_model = recommendation["selected_model"].iloc[0]

model_row = summary[summary["model"] == selected_model].iloc[0]
pred_row = pred_check[pred_check["model"] == selected_model].iloc[0]

best_model_path = file_manifest[
    (file_manifest["model"] == selected_model) &
    (file_manifest["file_type"] == "best_model")
]["path"].iloc[0]

tokenizer_path = file_manifest[
    (file_manifest["model"] == selected_model) &
    (file_manifest["file_type"] == "tokenizer")
]["path"].iloc[0]

if not Path(tokenizer_path).exists():
    tokenizer_path = best_model_path

shap_config = {
    "selected_model": selected_model,
    "best_model_path": best_model_path,
    "tokenizer_path": tokenizer_path,
    "max_length": 512,
    "label_map": {
        "0": "SAFE",
        "1": "INJECTION"
    },
    "text_column": pred_row["text_col"],
    "true_label_column": pred_row["true_col"],
    "predicted_label_column": pred_row["pred_col"],
    "safe_probability_column": pred_row["safe_prob_col"],
    "injection_probability_column": pred_row["inject_prob_col"],
    "main_reason": "Selected for SHAP because it has the strongest test/security performance."
}

web_config = {
    "selected_model": selected_model,
    "best_model_path": best_model_path,
    "tokenizer_path": tokenizer_path,
    "max_length": 512,
    "id2label": {
        "0": "SAFE",
        "1": "INJECTION"
    },
    "label2id": {
        "SAFE": 0,
        "INJECTION": 1
    },
    "output_labels": ["SAFE", "INJECTION"],
    "safe_class_id": 0,
    "injection_class_id": 1
}

selected_summary = pd.DataFrame([
    {
        "selected_model": selected_model,
        "validation_accuracy": model_row["validation_accuracy"],
        "validation_f1": model_row["validation_f1"],
        "test_accuracy": model_row["test_accuracy"],
        "test_precision": model_row["test_precision"],
        "test_recall": model_row["test_recall"],
        "test_f1": model_row["test_f1"],
        "test_auc_roc": model_row["test_auc_roc"],
        "false_positives": model_row["fp"],
        "false_negatives": model_row["fn"],
        "total_errors": model_row["total_errors"],
        "best_model_path": best_model_path,
        "tokenizer_path": tokenizer_path,
        "best_model_found": Path(best_model_path).exists(),
        "tokenizer_found": Path(tokenizer_path).exists()
    }
])

shap_config_file = config_dir / "selected_model_shap_config.json"
web_config_file = config_dir / "web_app_model_config.json"
selected_summary_file = table_dir / "selected_model_summary.csv"

with open(shap_config_file, "w") as f:
    json.dump(shap_config, f, indent=4)

with open(web_config_file, "w") as f:
    json.dump(web_config, f, indent=4)

selected_summary.to_csv(selected_summary_file, index=False)

check_df = pd.DataFrame([
    {"item": "selected_model", "value": selected_model},
    {"item": "best_model_found", "value": Path(best_model_path).exists()},
    {"item": "tokenizer_found", "value": Path(tokenizer_path).exists()},
    {"item": "shap_config_saved", "value": shap_config_file.exists()},
    {"item": "web_config_saved", "value": web_config_file.exists()},
    {"item": "summary_saved", "value": selected_summary_file.exists()}
])

display(selected_summary)
display(check_df)
print("Saved:", shap_config_file)
print("Saved:", web_config_file)
print("Saved:", selected_summary_file)

# Part 12 - Visualization

In [ ]:

summary = pd.read_csv(table_dir / "model_comparison_summary.csv")
all_logs = pd.read_csv(table_dir / "all_training_logs.csv")

figure_rows = []

def save_bar(df, x_col, y_col, title, ylabel, file_name):
    file_path = fig_dir / file_name

    plt.figure(figsize=(7, 5))
    plt.bar(df[x_col], df[y_col])
    plt.title(title)
    plt.xlabel("Model")
    plt.ylabel(ylabel)
    plt.tight_layout()
    plt.savefig(file_path, dpi=300)
    plt.show()

    figure_rows.append({
        "figure": file_name,
        "path": str(file_path),
        "saved": file_path.exists()
    })


save_bar(
    summary,
    "model",
    "validation_f1",
    "Validation F1-score Comparison",
    "Validation F1-score",
    "validation_f1_comparison.png"
)

save_bar(
    summary,
    "model",
    "test_f1",
    "Test F1-score Comparison",
    "Test F1-score",
    "test_f1_comparison.png"
)

save_bar(
    summary,
    "model",
    "test_auc_roc",
    "Test AUC-ROC Comparison",
    "Test AUC-ROC",
    "test_auc_roc_comparison.png"
)

save_bar(
    summary,
    "model",
    "fn",
    "False Negative Comparison",
    "False Negatives",
    "false_negative_comparison.png"
)

fp_fn = summary[["model", "fp", "fn"]].copy()
fp_fn_file = fig_dir / "false_positive_false_negative_comparison.png"

plt.figure(figsize=(7, 5))
x = np.arange(len(fp_fn["model"]))
width = 0.35

plt.bar(x - width/2, fp_fn["fp"], width, label="False Positive")
plt.bar(x + width/2, fp_fn["fn"], width, label="False Negative")

plt.xticks(x, fp_fn["model"])
plt.title("False Positive and False Negative Comparison")
plt.xlabel("Model")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.savefig(fp_fn_file, dpi=300)
plt.show()

figure_rows.append({
    "figure": "false_positive_false_negative_comparison.png",
    "path": str(fp_fn_file),
    "saved": fp_fn_file.exists()
})


loss_file = fig_dir / "training_loss_comparison.png"

plt.figure(figsize=(8, 5))

for model in models:
    model_log = all_logs[all_logs["model"] == model].copy()

    if "epoch" in model_log.columns and "loss" in model_log.columns:
        model_log = model_log[model_log["loss"].notna()]
        plt.plot(model_log["epoch"], model_log["loss"], marker="o", label=model)

plt.title("Training Loss Comparison")
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.legend()
plt.tight_layout()
plt.savefig(loss_file, dpi=300)
plt.show()

figure_rows.append({
    "figure": "training_loss_comparison.png",
    "path": str(loss_file),
    "saved": loss_file.exists()
})


f1_col = None

for col in ["eval_f1", "validation_f1", "val_f1", "f1"]:
    if col in all_logs.columns:
        f1_col = col
        break

f1_epoch_file = fig_dir / "validation_f1_by_epoch_comparison.png"

plt.figure(figsize=(8, 5))

if f1_col is not None:
    for model in models:
        model_log = all_logs[all_logs["model"] == model].copy()
        model_log = model_log[model_log[f1_col].notna()]

        if "epoch" in model_log.columns and len(model_log) > 0:
            plt.plot(model_log["epoch"], model_log[f1_col], marker="o", label=model)

plt.title("Validation F1-score by Epoch")
plt.xlabel("Epoch")
plt.ylabel("Validation F1-score")
plt.legend()
plt.tight_layout()
plt.savefig(f1_epoch_file, dpi=300)
plt.show()

figure_rows.append({
    "figure": "validation_f1_by_epoch_comparison.png",
    "path": str(f1_epoch_file),
    "saved": f1_epoch_file.exists()
})


figure_check = pd.DataFrame(figure_rows)

figure_file = table_dir / "figure_file_check.csv"
figure_check.to_csv(figure_file, index=False)

display(figure_check)
print("Saved:", figure_file)